# 01. Importação das bibliotecas

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Configuração visual dos gráficos
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

# 2. Localização e marcação do dataset

In [ ]:
DATA_DIR = Path("..") / "data" / "raw"

csv_files = sorted(DATA_DIR.glob("user_*.csv"))

print(f"Quantidade de CSVs encontrados: {len(csv_files)}")

for file in csv_files:
    print("-", file)

if len(csv_files) != 5:
    raise FileNotFoundError(
        "Era esperado encontrar 5 arquivos CSV em data/raw: "
        "user_a.csv, user_b.csv, user_c.csv, user_d.csv, user_e.csv."
    )

# 3. Leitura dos arquivos na pasta consolidada

In [ ]:
dfs = []

for file in csv_files:
    temp = pd.read_csv(file)
    temp["usuario"] = file.stem
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print("Dimensão final do dataset:", df.shape)
display(df.head())

In [ ]:
display(df.info())

display(df.describe().T)

In [ ]:
target_col = "Values"
user_col = "usuario"

feature_cols = [col for col in df.columns if col not in [target_col, user_col]]

print("Coluna alvo:", target_col)
print("Coluna de usuário:", user_col)
print("Quantidade de features:", len(feature_cols))
print("Features:")
display(feature_cols)

In [ ]:
feature_metadata = []

for col in feature_cols:
    parts = col.split(".")
    if len(parts) == 3:
        _, channel, band = parts
    else:
        channel, band = "desconhecido", "desconhecida"
    feature_metadata.append({"feature": col, "canal": channel, "banda": band})

feature_meta = pd.DataFrame(feature_metadata)

print("Canais encontrados:", sorted(feature_meta["canal"].unique()))
print("Bandas encontradas:", sorted(feature_meta["banda"].unique()))

display(feature_meta)

In [ ]:
quality_summary = pd.DataFrame({
    "coluna": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "nulos": df.isna().sum().values,
    "nulos_%": (df.isna().mean().values * 100),
    "unicos": df.nunique().values
})

display(quality_summary)

print("Total de valores nulos:", df.isna().sum().sum())
print("Total de linhas duplicadas:", df.duplicated().sum())
print("Existe infinito nas features?", np.isinf(df[feature_cols].to_numpy()).any())
print("Classes encontradas:", sorted(df[target_col].unique()))

In [ ]:
user_counts = df[user_col].value_counts().sort_index()

display(user_counts.to_frame("quantidade_amostras"))

plt.figure(figsize=(10, 5))
ax = sns.barplot(x=user_counts.index, y=user_counts.values)
ax.set_title("Quantidade de amostras por usuário")
ax.set_xlabel("Usuário")
ax.set_ylabel("Quantidade de amostras")

for i, v in enumerate(user_counts.values):
    ax.text(i, v, str(v), ha="center", va="bottom")

plt.show()

In [ ]:
class_counts = df[target_col].value_counts().sort_index()
class_percent = (class_counts / class_counts.sum() * 100).round(2)

class_distribution = pd.DataFrame({
    "classe": class_counts.index,
    "quantidade": class_counts.values,
    "percentual": class_percent.values
})

display(class_distribution)

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=class_distribution, x="classe", y="quantidade")
ax.set_title("Distribuição geral das classes")
ax.set_xlabel("Classe")
ax.set_ylabel("Quantidade de amostras")

for i, row in class_distribution.iterrows():
    ax.text(i, row["quantidade"], f'{row["percentual"]:.2f}%', ha="center", va="bottom")

plt.show()

In [ ]:
user_class_counts = pd.crosstab(df[user_col], df[target_col])
user_class_percent = pd.crosstab(df[user_col], df[target_col], normalize="index") * 100

display(user_class_counts)
display(user_class_percent.round(2))

plt.figure(figsize=(10, 6))
user_class_counts.plot(kind="bar", stacked=False, figsize=(12, 6))
plt.title("Distribuição das classes por usuário")
plt.xlabel("Usuário")
plt.ylabel("Quantidade de amostras")
plt.xticks(rotation=0)
plt.legend(title="Classe")
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(user_class_percent, annot=True, fmt=".1f", cmap="Blues")
plt.title("Percentual de classes por usuário")
plt.xlabel("Classe")
plt.ylabel("Usuário")
plt.show()

In [ ]:
feature_stats = df[feature_cols].agg(["min", "mean", "median", "std", "max"]).T
feature_stats["range"] = feature_stats["max"] - feature_stats["min"]

display(feature_stats.sort_values("std", ascending=False))

plt.figure(figsize=(14, 6))
sns.boxplot(data=df[feature_cols], orient="h")
plt.title("Boxplot das features EEG")
plt.xlabel("Valor")
plt.ylabel("Feature")
plt.show()

In [ ]:
df_long = df.melt(
    id_vars=[target_col, user_col],
    value_vars=feature_cols,
    var_name="feature",
    value_name="potencia"
)

df_long = df_long.merge(feature_meta, on="feature", how="left")

display(df_long.head())
print("Formato longo:", df_long.shape)

In [ ]:
band_summary = (
    df_long
    .groupby("banda")["potencia"]
    .agg(["mean", "median", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

display(band_summary)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_long, x="banda", y="potencia", estimator=np.mean, errorbar=("ci", 95))
plt.title("Potência média por banda de frequência")
plt.xlabel("Banda")
plt.ylabel("Potência média")
plt.show()

In [ ]:
channel_summary = (
    df_long
    .groupby("canal")["potencia"]
    .agg(["mean", "median", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

display(channel_summary)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_long, x="canal", y="potencia", estimator=np.mean, errorbar=("ci", 95))
plt.title("Potência média por canal EEG")
plt.xlabel("Canal")
plt.ylabel("Potência média")
plt.show()

In [ ]:
channel_band_mean = (
    df_long
    .groupby(["canal", "banda"])["potencia"]
    .mean()
    .reset_index()
    .pivot(index="canal", columns="banda", values="potencia")
)

display(channel_band_mean)

plt.figure(figsize=(10, 6))
sns.heatmap(channel_band_mean, annot=True, fmt=".1f", cmap="viridis")
plt.title("Potência média por canal e banda")
plt.xlabel("Banda")
plt.ylabel("Canal")
plt.show()

In [ ]:
class_band_mean = (
    df_long
    .groupby([target_col, "banda"])["potencia"]
    .mean()
    .reset_index()
)

display(class_band_mean)

plt.figure(figsize=(12, 6))
sns.barplot(data=class_band_mean, x="banda", y="potencia", hue=target_col)
plt.title("Potência média por banda e classe")
plt.xlabel("Banda")
plt.ylabel("Potência média")
plt.legend(title="Classe")
plt.show()

In [ ]:
class_channel_mean = (
    df_long
    .groupby([target_col, "canal"])["potencia"]
    .mean()
    .reset_index()
)

display(class_channel_mean)

plt.figure(figsize=(12, 6))
sns.barplot(data=class_channel_mean, x="canal", y="potencia", hue=target_col)
plt.title("Potência média por canal e classe")
plt.xlabel("Canal")
plt.ylabel("Potência média")
plt.legend(title="Classe")
plt.show()

In [ ]:
for cls in sorted(df[target_col].unique()):
    temp = (
        df_long[df_long[target_col] == cls]
        .groupby(["canal", "banda"])["potencia"]
        .mean()
        .reset_index()
        .pivot(index="canal", columns="banda", values="potencia")
    )

    plt.figure(figsize=(10, 5))
    sns.heatmap(temp, annot=True, fmt=".1f", cmap="viridis")
    plt.title(f"Potência média por canal e banda — Classe {cls}")
    plt.xlabel("Banda")
    plt.ylabel("Canal")
    plt.show()

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(feature_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for ax, col in zip(axes, feature_cols):
    sns.histplot(df[col], bins=40, kde=True, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("Valor")
    ax.set_ylabel("Frequência")

for ax in axes[len(feature_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
top_var_features = df[feature_cols].var().sort_values(ascending=False).head(6).index.tolist()

print("Features com maior variância:")
display(top_var_features)

for col in top_var_features:
    plt.figure(figsize=(10, 5))
    sns.kdeplot(data=df, x=col, hue=target_col, fill=True, common_norm=False, alpha=0.35)
    plt.title(f"Distribuição por classe — {col}")
    plt.xlabel(col)
    plt.ylabel("Densidade")
    plt.show()

In [ ]:
for col in top_var_features:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=target_col, y=col)
    plt.title(f"Boxplot por classe — {col}")
    plt.xlabel("Classe")
    plt.ylabel(col)
    plt.show()

In [ ]:
corr = df[feature_cols].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True)
plt.title("Matriz de correlação entre features")
plt.show()

# Pares mais correlacionados, ignorando a diagonal
corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["feature_1", "feature_2", "correlacao"]
corr_pairs["correlacao_abs"] = corr_pairs["correlacao"].abs()

display(corr_pairs.sort_values("correlacao_abs", ascending=False).head(15))

In [ ]:
corr_long = corr_pairs.merge(feature_meta.rename(columns={"feature": "feature_1", "banda": "banda_1", "canal": "canal_1"}), on="feature_1")
corr_long = corr_long.merge(feature_meta.rename(columns={"feature": "feature_2", "banda": "banda_2", "canal": "canal_2"}), on="feature_2")

same_band_corr = (
    corr_long[corr_long["banda_1"] == corr_long["banda_2"]]
    .groupby("banda_1")["correlacao_abs"]
    .mean()
    .sort_values(ascending=False)
)

display(same_band_corr.to_frame("correlacao_abs_media_mesma_banda"))

plt.figure(figsize=(10, 5))
sns.barplot(x=same_band_corr.index, y=same_band_corr.values)
plt.title("Correlação absoluta média entre features da mesma banda")
plt.xlabel("Banda")
plt.ylabel("Correlação absoluta média")
plt.show()

In [ ]:
df_time = df.copy()
df_time["sample_idx"] = df_time.groupby(user_col).cumcount()

# Features exemplo: uma por banda, usando o canal AF3
example_features = [col for col in feature_cols if col.startswith("POW.AF3.")]

print("Features exemplo para análise temporal:")
display(example_features)

for user in sorted(df_time[user_col].unique()):
    user_data = df_time[df_time[user_col] == user]

    plt.figure(figsize=(14, 5))
    for col in example_features:
        plt.plot(user_data["sample_idx"], user_data[col], label=col, alpha=0.75)

    plt.title(f"Variação temporal aproximada das bandas no canal AF3 — {user}")
    plt.xlabel("Índice da amostra")
    plt.ylabel("Potência")
    plt.legend()
    plt.show()

In [ ]:
window = 100

for user in sorted(df_time[user_col].unique()):
    user_data = df_time[df_time[user_col] == user].copy()
    col = "POW.AF3.Alpha"

    user_data[f"{col}_rolling"] = user_data[col].rolling(window=window, min_periods=1).mean()

    plt.figure(figsize=(14, 5))
    plt.plot(user_data["sample_idx"], user_data[col], alpha=0.25, label="Original")
    plt.plot(user_data["sample_idx"], user_data[f"{col}_rolling"], label=f"Média móvel ({window})")
    plt.title(f"Média móvel da feature {col} — {user}")
    plt.xlabel("Índice da amostra")
    plt.ylabel("Potência")
    plt.legend()
    plt.show()

In [ ]:
X = df[feature_cols].copy()
y = df[target_col].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "classe": y.astype(str),
    "usuario": df[user_col].values
})

print("Variância explicada por componente:", pca.explained_variance_ratio_)
print("Variância explicada acumulada:", pca.explained_variance_ratio_.sum())

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="classe", alpha=0.5, s=20)
plt.title("PCA 2D colorido por classe")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Classe")
plt.show()

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="usuario", alpha=0.5, s=20)
plt.title("PCA 2D colorido por usuário")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Usuário")
plt.show()

In [ ]:
pca_loadings = pd.DataFrame(
    pca.components_.T,
    columns=["PC1", "PC2"],
    index=feature_cols
)

display(pca_loadings.assign(
    PC1_abs=pca_loadings["PC1"].abs(),
    PC2_abs=pca_loadings["PC2"].abs()
).sort_values("PC1_abs", ascending=False).head(10))

plt.figure(figsize=(10, 6))
pca_loadings["PC1"].sort_values().plot(kind="barh")
plt.title("Pesos das features no PC1")
plt.xlabel("Peso")
plt.ylabel("Feature")
plt.show()

plt.figure(figsize=(10, 6))
pca_loadings["PC2"].sort_values().plot(kind="barh")
plt.title("Pesos das features no PC2")
plt.xlabel("Peso")
plt.ylabel("Feature")
plt.show()

In [ ]:
RUN_TSNE = True
TSNE_SAMPLE_SIZE = 5000

if RUN_TSNE:
    sample_df = df.sample(n=min(TSNE_SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE)
    X_sample = sample_df[feature_cols]
    y_sample = sample_df[target_col]

    X_sample_scaled = StandardScaler().fit_transform(X_sample)

    tsne = TSNE(
        n_components=2,
        random_state=RANDOM_STATE,
        perplexity=30,
        learning_rate="auto",
        init="pca"
    )

    X_tsne = tsne.fit_transform(X_sample_scaled)

    tsne_df = pd.DataFrame({
        "TSNE1": X_tsne[:, 0],
        "TSNE2": X_tsne[:, 1],
        "classe": y_sample.astype(str).values,
        "usuario": sample_df[user_col].values
    })

    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=tsne_df, x="TSNE1", y="TSNE2", hue="classe", alpha=0.65, s=25)
    plt.title("t-SNE 2D colorido por classe")
    plt.xlabel("TSNE1")
    plt.ylabel("TSNE2")
    plt.legend(title="Classe")
    plt.show()

    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=tsne_df, x="TSNE1", y="TSNE2", hue="usuario", alpha=0.65, s=25)
    plt.title("t-SNE 2D colorido por usuário")
    plt.xlabel("TSNE1")
    plt.ylabel("TSNE2")
    plt.legend(title="Usuário")
    plt.show()
else:
    print("t-SNE desativado. Mude RUN_TSNE para True se quiser executar.")

In [ ]:
outlier_rows = []

for col in feature_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_rows.append({
        "feature": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "limite_inferior": lower,
        "limite_superior": upper,
        "qtd_outliers": outliers,
        "percentual_outliers": outliers / len(df) * 100
    })

outlier_summary = pd.DataFrame(outlier_rows).sort_values("percentual_outliers", ascending=False)

display(outlier_summary)

plt.figure(figsize=(12, 6))
sns.barplot(data=outlier_summary, y="feature", x="percentual_outliers")
plt.title("Percentual de possíveis outliers por feature — método IQR")
plt.xlabel("% de outliers")
plt.ylabel("Feature")
plt.show()

In [ ]:
user_band_mean = (
    df_long
    .groupby([user_col, "banda"])["potencia"]
    .mean()
    .reset_index()
)

display(user_band_mean)

plt.figure(figsize=(12, 6))
sns.barplot(data=user_band_mean, x="banda", y="potencia", hue=user_col)
plt.title("Potência média por banda e usuário")
plt.xlabel("Banda")
plt.ylabel("Potência média")
plt.legend(title="Usuário")
plt.show()

In [ ]:
user_channel_mean = (
    df_long
    .groupby([user_col, "canal"])["potencia"]
    .mean()
    .reset_index()
)

display(user_channel_mean)

plt.figure(figsize=(12, 6))
sns.barplot(data=user_channel_mean, x="canal", y="potencia", hue=user_col)
plt.title("Potência média por canal e usuário")
plt.xlabel("Canal")
plt.ylabel("Potência média")
plt.legend(title="Usuário")
plt.show()

In [ ]:
summary_text = f'''
Resumo da EDA:
- O dataset possui {df.shape[0]:,} amostras e {df.shape[1]} colunas, considerando a coluna de usuário criada no carregamento.
- Foram identificados {df[user_col].nunique()} usuários: {", ".join(sorted(df[user_col].unique()))}.
- A variável alvo é '{target_col}', com classes {sorted(df[target_col].unique())}.
- Existem {len(feature_cols)} features numéricas extraídas de EEG.
- Não foram encontrados {df.isna().sum().sum()} valores ausentes.
- Foram encontradas {df.duplicated().sum()} linhas duplicadas.
- Os canais identificados foram: {", ".join(sorted(feature_meta["canal"].unique()))}.
- As bandas identificadas foram: {", ".join(sorted(feature_meta["banda"].unique()))}.
'''

print(summary_text)